In [1]:
# ==============================================================================
# REPOSITORY: urban-heat-island-analytics
# SCRIPT: 01_uhi_preprocessing_pipeline.py
# DESCRIPTION: Pipeline agnóstico de extracción multitemporal (Sentinel-2,
#              Landsat 8/9, ERA5-Land, WorldPop), tratamiento de nulos y clustering (K-Means).
# ==============================================================================

import calendar
import ee
import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from google.colab import drive

# ------------------------------------------------------------------------------
# 1. PARÁMETROS GLOBALES
# ------------------------------------------------------------------------------
START_YEAR = 2016
END_YEAR = 2026
CLOUD_FILTER_PCT = 60

# URL o ruta local a tu GeoJSON / Shapefile con las geometrías de las zonas
ROI_GEOJSON_URL = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/arrondissements/exports/geojson"

# ------------------------------------------------------------------------------
# 2. INICIALIZACIÓN DE EARTH ENGINE Y CARGA DE GEOMETRÍAS
# ------------------------------------------------------------------------------
GEE_PROJECT_ID = 'titanium-vision-322421'

try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print("¡Conectado exitosamente a Google Earth Engine!")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print("Autenticación completada y conectado a Earth Engine.")

response = requests.get(ROI_GEOJSON_URL)
geojson_raw = response.json()
roi_gdf = gpd.GeoDataFrame.from_features(geojson_raw["features"])

if 'c_ar' in roi_gdf.columns:
    roi_gdf['zone_id'] = roi_gdf['c_ar'].astype(int)
elif 'code' in roi_gdf.columns:
    roi_gdf['zone_id'] = roi_gdf['code'].astype(int)
else:
    roi_gdf['zone_id'] = roi_gdf.index + 1

roi_fc = ee.FeatureCollection(roi_gdf.__geo_interface__)
print(f"Cargadas {len(roi_gdf)} zonas de estudio para el análisis espacial.")

# ------------------------------------------------------------------------------
# 3. BUCLE DE EXTRACCIÓN MENSUAL (Fusión de Sensores y Demografía)
# ------------------------------------------------------------------------------
records_monthly = []
print(f"Iniciando extracción mensual ({START_YEAR} - {END_YEAR}) con filtro de nubes < {CLOUD_FILTER_PCT}%...")

for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        last_day = calendar.monthrange(year, month)[1]
        start_date = f"{year}-{month:02d}-01"
        end_date = f"{year}-{month:02d}-{last_day:02d}"

        # A. Sentinel-2: NDVI
        s2_col = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(roi_fc)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_FILTER_PCT))
            .select(['B8', 'B4'])
        )
        s2_dict = {}
        if s2_col.size().getInfo() > 0:
            s2_img = s2_col.median()
            ndvi_s2 = s2_img.normalizedDifference(['B8', 'B4']).rename('NDVI_S2')
            stats_s2 = ndvi_s2.reduceRegions(collection=roi_fc, reducer=ee.Reducer.mean(), scale=30).getInfo()
            for f in stats_s2.get('features', []):
                p = f['properties']
                z_id = p.get('zone_id', p.get('c_ar', p.get('arrondissement', None)))
                if z_id is not None:
                    s2_dict[int(z_id)] = p.get('mean')

        # B. Landsat 8/9: LST
        landsat_col = (
            ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
            .filterBounds(roi_fc)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt('CLOUD_COVER', CLOUD_FILTER_PCT))
            .select(['ST_B10'])
        )
        lst_dict = {}
        if landsat_col.size().getInfo() > 0:
            landsat_img = landsat_col.median()
            lst_celsius = landsat_img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST')
            stats_lst = lst_celsius.reduceRegions(collection=roi_fc, reducer=ee.Reducer.mean(), scale=30).getInfo()
            for f in stats_lst.get('features', []):
                p = f['properties']
                z_id = p.get('zone_id', p.get('c_ar', p.get('arrondissement', None)))
                if z_id is not None:
                    lst_dict[int(z_id)] = p.get('mean')

        # C. ERA5-Land: Reanálisis Climático
        era5_col = (
            ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
            .filterDate(start_date, end_date)
            .select(['temperature_2m', 'volumetric_soil_water_layer_1'])
        )
        era5_dict = {}
        if era5_col.size().getInfo() > 0:
            try:
                era5_mean = era5_col.mean()
                era5_img = era5_mean.select('temperature_2m').subtract(273.15).rename('T2M').addBands(
                    era5_mean.select('volumetric_soil_water_layer_1').rename('SOIL_MOISTURE')
                )
                stats_era5 = era5_img.reduceRegions(collection=roi_fc, reducer=ee.Reducer.mean(), scale=1000).getInfo()
                for f in stats_era5.get('features', []):
                    p = f['properties']
                    z_id = p.get('zone_id', p.get('c_ar', p.get('arrondissement', None)))
                    if z_id is not None:
                        era5_dict[int(z_id)] = {
                            'T2M_era5': p.get('T2M'),
                            'Soil_Moisture': p.get('SOIL_MOISTURE')
                        }
            except Exception:
                pass

        # D. WorldPop: Población estimada (Densidad Demográfica)
        pop_col = (
            ee.ImageCollection("WorldPop/GP/100m/pop")
            .filterBounds(roi_fc)
            .filter(ee.Filter.calendarRange(year, year, 'year'))
        )
        pop_dict = {}
        if pop_col.size().getInfo() > 0:
            pop_img = pop_col.sum()
        else:
            pop_img = ee.ImageCollection("WorldPop/GP/100m/pop").filterBounds(roi_fc).sort('system:time_start', False).first()
        try:
            stats_pop = pop_img.reduceRegions(collection=roi_fc, reducer=ee.Reducer.sum(), scale=100).getInfo()
            for f in stats_pop.get('features', []):
                p = f['properties']
                z_id = p.get('zone_id', p.get('c_ar', p.get('arrondissement', None)))
                if z_id is not None:
                    pop_dict[int(z_id)] = p.get('sum')
        except Exception:
            pass

        # Compilación de registros por cada zona
        for zone_id in roi_gdf['zone_id']:
            records_monthly.append({
                'zone_id': zone_id,
                'year': year,
                'month': month,
                'NDVI_S2_10m': s2_dict.get(zone_id, None),
                'LST_Landsat_30m': lst_dict.get(zone_id, None),
                'T2M_air_era5': era5_dict.get(zone_id, {}).get('T2M_era5', None),
                'Soil_Moisture_era5': era5_dict.get(zone_id, {}).get('Soil_Moisture', None),
                'Population_Count': pop_dict.get(zone_id, None)
            })

print("¡Extracción espacial completada!")

# ------------------------------------------------------------------------------
# 4. TRATAMIENTO DE NULOS, BANDERAS DE AUDITORÍA E INTERPOLACIÓN
# ------------------------------------------------------------------------------
df_timeseries = pd.DataFrame(records_monthly)
df_timeseries = df_timeseries.sort_values(by=['zone_id', 'year', 'month']).reset_index(drop=True)

# Se integra la nueva variable poblacional para la limpieza
num_cols = ['NDVI_S2_10m', 'LST_Landsat_30m', 'T2M_air_era5', 'Soil_Moisture_era5', 'Population_Count']
for col in num_cols:
    df_timeseries[col] = pd.to_numeric(df_timeseries[col], errors='coerce')

df_timeseries['NDVI_proyectado'] = df_timeseries['NDVI_S2_10m'].isna()
df_timeseries['LST_proyectado'] = df_timeseries['LST_Landsat_30m'].isna()

for col in num_cols:
    df_timeseries[col] = df_timeseries.groupby('zone_id')[col].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    df_timeseries[col] = df_timeseries[col].fillna(df_timeseries[col].mean())

# ------------------------------------------------------------------------------
# 5. MACHINE LEARNING: CLUSTERING DE VULNERABILIDAD TÉRMICA Y SOCIAL (K-Means)
# ------------------------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_timeseries[num_cols])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_timeseries['cluster_vulnerabilidad'] = kmeans.fit_predict(X_scaled)

# ------------------------------------------------------------------------------
# 6. EXPORTACIÓN A GOOGLE DRIVE
# ------------------------------------------------------------------------------
drive.mount('/content/drive', force_remount=False)
output_path = '/content/drive/My Drive/uhi_timeseries_master.csv'
df_timeseries.to_csv(output_path, index=False)

print(f"\n¡Dataset maestro guardado exitosamente en Google Drive: {output_path}!")
display(df_timeseries.head(15))

Autenticación completada y conectado a Earth Engine.
Cargadas 20 zonas de estudio para el análisis espacial.
Iniciando extracción mensual (2016 - 2026) con filtro de nubes < 60%...


/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for WorldPop/GP/100m/pop! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by WorldPop/GP/100m/R2025A/v1/pop

Learn more: https://developers.google.com/earth-engine/datasets/catalog/WorldPop_GP_100m_pop

  warnings.warn(warning, category=DeprecationWarning)


¡Extracción espacial completada!
Mounted at /content/drive

¡Dataset maestro guardado exitosamente en Google Drive: /content/drive/My Drive/uhi_timeseries_master.csv!


,zone_id,year,month,NDVI_S2_10m,LST_Landsat_30m,T2M_air_era5,Soil_Moisture_era5,Population_Count,NDVI_proyectado,LST_proyectado,cluster_vulnerabilidad
0,1,2016,1,0.040241,13.756242,5.139965,0.391005,15242.058248,False,True,2
1,1,2016,2,0.059874,13.756242,5.581032,0.396679,15242.058248,True,True,2
2,1,2016,3,0.079507,13.756242,6.266839,0.383469,15242.058248,True,True,2
3,1,2016,4,0.099139,13.756242,9.208963,0.378534,15242.058248,True,False,2
4,1,2016,5,0.118772,26.035451,13.918360,0.365741,15242.058248,False,False,2
5,1,2016,6,0.114812,26.355492,16.806885,0.379977,15242.058248,True,False,2
6,1,2016,7,0.110852,29.443665,19.490548,0.245529,15242.058248,True,True,0
7,1,2016,8,0.106892,32.531837,20.800734,0.175152,15242.058248,False,False,0
8,1,2016,9,0.105284,21.658169,18.961972,0.174751,15242.058248,False,False,0
9,1,2016,10,0.101730,17.233242,11.399430,0.243769,15242.058248,True,False,2
